In [ ]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

In [ ]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <iostream> // for std::ostream
#include <vector>

namespace csc485b{
namespace a2{

using node_t = int;
using edge_t = int2;

using edge_list_t = std::vector< edge_t >;
using node_list_t = std::vector< node_t >;

} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [ ]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <algorithm> // std::min
#include <cassert>   // for assert()
#include <cmath>     // std::log1p, std::floor
#include <cstddef>   // std::size_t type
#include <random>    // for std::mt19937, std::uniform_real_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2 {

/**
 * Generates and returns a vector of random edges
 * for a graph `G=(V,E)` with `n=|V|=n` and expected `m=|E|`.
 * Referred to as an Erdős-Rényi graph. `m` counts directed edges,
 * i.e., every undirected edge {u,v} appears as both (u,v) and (v,u).
 *
 * @see https://networkx.org/documentation/stable/reference/generated/networkx.generators.random_graphs.fast_gnp_random_graph.html#networkx.generators.random_graphs.fast_gnp_random_graph
 */
edge_list_t generate_graph( std::size_t n, std::size_t m )
{
    assert( "At most n(n-1) edges in a simple graph" && ( n < 2 || m < n * ( n - 1 ) ) );

    edge_list_t random_edges;
    if( n < 2 || m == 0 ) return random_edges;

    double const probability = static_cast< double >( m ) / static_cast< double >( n * ( n - 1 ) );

    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_real_distribution
    std::size_t random_seed = 20241008;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_real_distribution< double > uniform( 0.0, 1.0 );

    random_edges.reserve( m + m / 4 + 16 );

    // Walk over all pairs (w, v) with w < v in order, jumping ahead by a
    // geometrically-distributed gap each time instead of rolling for every pair.
    double const log_q = std::log1p( -probability ); // log(1 - p) < 0
    long long const N  = static_cast< long long >( n );
    long long v = 1, w = -1;
    while( v < N )
    {
        double const r    = uniform( rng );
        double const skip = std::floor( std::log1p( -r ) / log_q );
        w += 1 + static_cast< long long >( std::min( skip, static_cast< double >( N * N ) ) );
        while( w >= v && v < N )
        {
            w -= v;
            ++v;
        }
        if( v < N )
        {
            random_edges.push_back( make_int2( static_cast< node_t >( w ), static_cast< node_t >( v ) ) );
            random_edges.push_back( make_int2( static_cast< node_t >( v ), static_cast< node_t >( w ) ) );
        }
    }

    return random_edges;
}

} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "dense_graph.h"
/**
 * The file in which you will implement your DenseGraph GPU solutions!
 */
#pragma once

#include <algorithm> // std::min, std::swap
#include <cstddef>   // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A DenseGraph is optimised for a graph in which the number of edges
 * is close to n(n-1). It is represented using an adjacency matrix.
 * Stored as a 1d row-major array: cell (i,j) lives at adjacencyMatrix[i*n + j].
 */
struct DenseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  node_t * adjacencyMatrix; /** Pointer to an n x n adj. matrix */

  /** Returns number of cells in the adjacency matrix. */
  __device__ __host__ __forceinline__
  std::size_t matrix_size() const { return n * n; }
};

/** Which two-hop kernel to use (the naive one is kept as a baseline for comparison). */
enum class DenseAlgo { Naive, Tiled };


namespace gpu {

// Tile configuration for the shared-memory GEMM.
// A block of 32 x 8 threads computes one 32 x 32 tile of the output;
// each thread accumulates 4 output cells (rows ty, ty+8, ty+16, ty+24).
constexpr int TILE          = 32;
constexpr int TILE_ROWS     = 8;
constexpr int ROWS_PER_THREAD = TILE / TILE_ROWS;

/**
 * Constructs a DenseGraph from an input edge list of m edges.
 * One thread per edge (grid-stride loop so any grid size works).
 *
 * @pre The pointers in DenseGraph g have already been allocated.
 * @pre The matrix has been zeroed (done by the host wrapper below).
 *
 * Two threads may write the same cell only if the edge list contained a
 * duplicate edge; they write the same value, so this is a benign race.
 */
__global__
void build_graph( DenseGraph g, edge_t const * edge_list, std::size_t m )
{
    std::size_t const stride = static_cast< std::size_t >( gridDim.x ) * blockDim.x;
    for( std::size_t i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x; i < m; i += stride )
    {
        edge_t const e = edge_list[ i ];
        g.adjacencyMatrix[ static_cast< std::size_t >( e.x ) * g.n + e.y ] = 1;
    }
}

/**
 * Naive baseline: one thread per output cell, reading straight from global memory.
 * out[i][j] = 1 iff i != j and (A*A)[i][j] > 0.
 *
 * Must write to a separate output buffer: squaring in place would let threads
 * read cells that other threads have already overwritten.
 */
__global__
void two_hop_reachability_naive( DenseGraph g, node_t * out )
{
    std::size_t const n   = g.n;
    std::size_t const col = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x;
    std::size_t const row = blockIdx.y * static_cast< std::size_t >( blockDim.y ) + threadIdx.y;
    if( row >= n || col >= n ) return;

    node_t const * A = g.adjacencyMatrix;
    node_t sum = 0;
    for( std::size_t k = 0; k < n; ++k )
    {
        sum += A[ row * n + k ] * A[ k * n + col ];
    }
    out[ row * n + col ] = ( row != col && sum > 0 ) ? 1 : 0;
}

/**
 * Tiled GEMM (C = A * A) with shared memory, then clamp to {0,1} and zero the diagonal.
 * Launch with blockDim = (TILE, TILE_ROWS) and gridDim = (ceil(n/TILE), ceil(n/TILE)).
 *
 * Synchronisation used:
 *   - __syncthreads()      : block-level barrier after cooperatively loading a tile.
 *   - __syncthreads_and()  : block-level barrier + vote. Because we only need a boolean
 *                            answer, once every cell in this block's output tile is already
 *                            > 0 no further tiles can change the result, so the whole block
 *                            stops early. (Big win on dense graphs; never hurts correctness.)
 *
 * Memory access:
 *   - Global loads are coalesced (consecutive tx -> consecutive columns).
 *   - As[r][k] is the same address for every thread in a warp (broadcast, no conflict),
 *     Bs[k][tx] is consecutive across the warp (no bank conflicts).
 */
__global__
void two_hop_reachability( DenseGraph g, node_t * out )
{
    __shared__ node_t As[ TILE ][ TILE ];
    __shared__ node_t Bs[ TILE ][ TILE ];

    int const n  = static_cast< int >( g.n );
    int const tx = threadIdx.x;
    int const ty = threadIdx.y;
    int const row0 = blockIdx.y * TILE;          // first output row of this tile
    int const col  = blockIdx.x * TILE + tx;     // this thread's output column

    node_t const * A = g.adjacencyMatrix;

    node_t acc[ ROWS_PER_THREAD ];
    #pragma unroll
    for( int r = 0; r < ROWS_PER_THREAD; ++r ) acc[ r ] = 0;

    int const num_tiles = ( n + TILE - 1 ) / TILE;
    for( int t = 0; t < num_tiles; ++t )
    {
        int const k0 = t * TILE;

        // cooperatively load one tile of A (rows of the output) and one of B (= A, columns of the output)
        #pragma unroll
        for( int r = 0; r < ROWS_PER_THREAD; ++r )
        {
            int const lr = ty + r * TILE_ROWS;

            int const a_row = row0 + lr, a_col = k0 + tx;
            As[ lr ][ tx ] = ( a_row < n && a_col < n ) ? A[ static_cast< std::size_t >( a_row ) * n + a_col ] : 0;

            int const b_row = k0 + lr;
            Bs[ lr ][ tx ] = ( b_row < n && col < n ) ? A[ static_cast< std::size_t >( b_row ) * n + col ] : 0;
        }
        __syncthreads();

        #pragma unroll
        for( int k = 0; k < TILE; ++k )
        {
            node_t const b = Bs[ k ][ tx ];
            #pragma unroll
            for( int r = 0; r < ROWS_PER_THREAD; ++r )
            {
                acc[ r ] += As[ ty + r * TILE_ROWS ][ k ] * b;
            }
        }

        // Is every cell of this output tile already decided? Out-of-range cells and the
        // diagonal (always forced to 0) count as decided.
        int decided = 1;
        #pragma unroll
        for( int r = 0; r < ROWS_PER_THREAD; ++r )
        {
            int const row = row0 + ty + r * TILE_ROWS;
            bool const cell_done = ( row >= n || col >= n || row == col || acc[ r ] > 0 );
            decided &= cell_done ? 1 : 0;
        }
        // Doubles as the barrier that protects As/Bs before the next tile is loaded.
        if( __syncthreads_and( decided ) ) break;
    }

    // epilogue: clamp to {0,1} and remove self loops
    #pragma unroll
    for( int r = 0; r < ROWS_PER_THREAD; ++r )
    {
        int const row = row0 + ty + r * TILE_ROWS;
        if( row < n && col < n )
        {
            out[ static_cast< std::size_t >( row ) * n + col ] = ( row != col && acc[ r ] > 0 ) ? 1 : 0;
        }
    }
}

} // namespace gpu


// ---------------------------------------------------------------------------------------
// Host-side entry points (these launch the kernels above)
// ---------------------------------------------------------------------------------------

/**
 * Builds the adjacency matrix on the device.
 * cudaMalloc does not zero memory, so the matrix is cleared first.
 */
inline void build_graph( DenseGraph & g, edge_t const * d_edges, std::size_t m )
{
    CUDA_CALL( cudaMemsetAsync( g.adjacencyMatrix, 0, sizeof( node_t ) * g.matrix_size() ) );
    if( m == 0 ) return;

    int const threads = 256;
    int const blocks  = static_cast< int >( std::min< std::size_t >( ( m + threads - 1 ) / threads, 1u << 16 ) );
    gpu::build_graph<<< blocks, threads >>>( g, d_edges, m );
    CHECK_ERROR( "dense build_graph" );
}

/**
 * Replaces g with its two-hop graph.
 * `scratch` must be a second n*n buffer; the result is written there and the two
 * pointers are swapped, so afterwards g.adjacencyMatrix points at the result and
 * `scratch` points at the old matrix (both still need to be freed by the caller).
 */
inline void two_hop_reachability( DenseGraph & g, node_t *& scratch, DenseAlgo algo = DenseAlgo::Tiled )
{
    if( g.n == 0 ) return;
    unsigned const tiles = static_cast< unsigned >( ( g.n + gpu::TILE - 1 ) / gpu::TILE );

    if( algo == DenseAlgo::Naive )
    {
        dim3 const block( 32, 8 );
        dim3 const grid( tiles, static_cast< unsigned >( ( g.n + 7 ) / 8 ) );
        gpu::two_hop_reachability_naive<<< grid, block >>>( g, scratch );
    }
    else
    {
        dim3 const block( gpu::TILE, gpu::TILE_ROWS );
        dim3 const grid( tiles, tiles );
        gpu::two_hop_reachability<<< grid, block >>>( g, scratch );
    }
    CHECK_ERROR( "dense two_hop_reachability" );

    std::swap( g.adjacencyMatrix, scratch );
}

} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "sparse_graph.h"
/**
 * The file in which you will implement your SparseGraph GPU solutions!
 */
#pragma once

#include <algorithm> // std::min, std::max, std::swap
#include <cstddef>   // std::size_t type
#include <cstdio>    // std::fprintf
#include <cstdlib>   // std::exit

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A SparseGraph is optimised for a graph in which the number of edges
 * is close to cn, for a small constanct c. It is represented in CSR format.
 */
struct SparseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  std::size_t m; /**< Number of edges in the graph. */
  node_t * neighbours_start_at; /** Pointer to an n+1 offset array */
  node_t * neighbours; /** Pointer to an m=|E| array of edge destinations */
};


namespace gpu {

constexpr int SCAN_BLOCK    = 1024; // elements (and threads) per block in the prefix scan
constexpr int TWO_HOP_BLOCK = 256;  // threads per block in the two-hop kernels

// ---------------------------------------------------------------------------------------
// Warp- and block-level scan primitives
// ---------------------------------------------------------------------------------------

/** Inclusive prefix sum across a warp using register shuffles (warp-level sync). */
__device__ __forceinline__
int warp_inclusive_scan( int x )
{
    int const lane = threadIdx.x & 31;
    #pragma unroll
    for( int d = 1; d < 32; d <<= 1 )
    {
        int const y = __shfl_up_sync( 0xffffffffu, x, d );
        if( lane >= d ) x += y;
    }
    return x;
}

/**
 * Exclusive prefix sum across a 1-D block (blockDim.x a multiple of 32, <= 1024).
 * Returns this thread's exclusive prefix and writes the block total to `total`.
 * Must be called by every thread of the block (it contains __syncthreads()).
 *
 * Step 1: each warp scans its own values with shuffles.
 * Step 2: warp 0 scans the 32 per-warp totals (stored in shared memory).
 * Step 3: each thread adds the total of all warps before it.
 */
__device__ __forceinline__
int block_exclusive_scan( int x, int & total )
{
    __shared__ int warp_sums[ 32 ];
    int const lane      = threadIdx.x & 31;
    int const warp      = threadIdx.x >> 5;
    int const num_warps = blockDim.x >> 5;

    int const inclusive = warp_inclusive_scan( x );
    if( lane == 31 ) warp_sums[ warp ] = inclusive;
    __syncthreads();

    if( warp == 0 )
    {
        int const s = ( lane < num_warps ) ? warp_sums[ lane ] : 0;
        warp_sums[ lane ] = warp_inclusive_scan( s );
    }
    __syncthreads();

    int const warp_prefix = ( warp > 0 ) ? warp_sums[ warp - 1 ] : 0;
    total = warp_sums[ num_warps - 1 ];
    __syncthreads(); // warp_sums may be reused by the next call

    return warp_prefix + inclusive - x;
}

/** Phase 1 of a device-wide scan: scan each block, record each block's total. */
__global__
void scan_blocks( node_t * data, std::size_t N, node_t * block_sums )
{
    std::size_t const i = blockIdx.x * static_cast< std::size_t >( SCAN_BLOCK ) + threadIdx.x;
    int const x = ( i < N ) ? data[ i ] : 0;
    int total;
    int const prefix = block_exclusive_scan( x, total );
    if( i < N ) data[ i ] = prefix;
    if( threadIdx.x == 0 && block_sums != nullptr ) block_sums[ blockIdx.x ] = total;
}

/** Phase 3 of a device-wide scan: add the (already scanned) block totals back in. */
__global__
void add_block_offsets( node_t * data, std::size_t N, node_t const * block_sums )
{
    std::size_t const i = blockIdx.x * static_cast< std::size_t >( SCAN_BLOCK ) + threadIdx.x;
    if( i < N ) data[ i ] += block_sums[ blockIdx.x ];
}

/** How much scratch space exclusive_scan() needs for N elements. */
inline std::size_t scan_scratch_size( std::size_t N )
{
    std::size_t total = 0;
    while( true )
    {
        std::size_t const blocks = ( N + SCAN_BLOCK - 1 ) / SCAN_BLOCK;
        if( blocks <= 1 ) break;
        total += blocks;
        N = blocks;
    }
    return std::max< std::size_t >( total, 1 );
}

/**
 * Device-wide exclusive prefix sum, in place. Grid-level synchronisation comes from
 * the kernel boundaries: block scans -> (recursive) scan of block totals -> add back.
 */
inline void exclusive_scan( node_t * d_data, std::size_t N, node_t * d_scratch )
{
    if( N == 0 ) return;
    std::size_t const blocks = ( N + SCAN_BLOCK - 1 ) / SCAN_BLOCK;
    if( blocks == 1 )
    {
        scan_blocks<<< 1, SCAN_BLOCK >>>( d_data, N, nullptr );
        CHECK_ERROR( "scan_blocks" );
        return;
    }
    scan_blocks<<< static_cast< unsigned >( blocks ), SCAN_BLOCK >>>( d_data, N, d_scratch );
    CHECK_ERROR( "scan_blocks" );
    exclusive_scan( d_scratch, blocks, d_scratch + blocks );
    add_block_offsets<<< static_cast< unsigned >( blocks ), SCAN_BLOCK >>>( d_data, N, d_scratch );
    CHECK_ERROR( "add_block_offsets" );
}

// ---------------------------------------------------------------------------------------
// Build: count -> scan -> scatter -> sort rows
// ---------------------------------------------------------------------------------------

/** degree[u] += 1 for every edge (u, v). Global atomics. */
__global__
void count_degrees( edge_t const * edge_list, std::size_t m, node_t * degree )
{
    std::size_t const stride = static_cast< std::size_t >( gridDim.x ) * blockDim.x;
    for( std::size_t i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x; i < m; i += stride )
    {
        atomicAdd( &degree[ edge_list[ i ].x ], 1 );
    }
}

/**
 * Places each edge (u, v) somewhere in row u of `unsorted` by grabbing a slot with an
 * atomic cursor. The order inside a row depends on thread timing, so rows get sorted next.
 */
__global__
void scatter_edges( edge_t const * edge_list, std::size_t m, node_t * cursor, node_t * unsorted )
{
    std::size_t const stride = static_cast< std::size_t >( gridDim.x ) * blockDim.x;
    for( std::size_t i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x; i < m; i += stride )
    {
        edge_t const e = edge_list[ i ];
        node_t const slot = atomicAdd( &cursor[ e.x ], 1 );
        unsorted[ slot ] = e.y;
    }
}

/**
 * Sorts every row by rank: the final position of v in row u is the number of
 * neighbours of u that are smaller than v. One thread per edge, no synchronisation
 * needed, deterministic output. Requires a simple graph (no duplicate edges).
 */
__global__
void rank_sort_rows( SparseGraph g, edge_t const * edge_list, std::size_t m, node_t const * unsorted )
{
    std::size_t const stride = static_cast< std::size_t >( gridDim.x ) * blockDim.x;
    for( std::size_t i = blockIdx.x * static_cast< std::size_t >( blockDim.x ) + threadIdx.x; i < m; i += stride )
    {
        edge_t const e   = edge_list[ i ];
        node_t const beg = g.neighbours_start_at[ e.x ];
        node_t const end = g.neighbours_start_at[ e.x + 1 ];
        node_t rank = 0;
        for( node_t p = beg; p < end; ++p ) rank += ( unsorted[ p ] < e.y ) ? 1 : 0;
        g.neighbours[ beg + rank ] = e.y;
    }
}

// ---------------------------------------------------------------------------------------
// Two-hop reachability
// ---------------------------------------------------------------------------------------

/**
 * One block handles one source node i at a time (grid-stride over nodes, so the grid
 * can be sized to exactly fill the GPU). The block keeps an n-bit bitmap in shared memory:
 *
 *   mark:    every warp takes a neighbour k of i, and its 32 lanes walk N(k) in parallel,
 *            setting bit j (j != i) with a shared-memory atomicOr. Duplicates (several
 *            k reaching the same j) just set the same bit again, so dedup is free.
 *   harvest: the block sweeps the bitmap word by word, and clears it for the next node.
 *            COUNT pass: popcount + block reduction -> number of 2-hop neighbours of i.
 *            WRITE pass: popcount + block exclusive scan gives each word's output position,
 *            so the neighbours come out already sorted.
 *
 * The output size isn't known in advance, so the host runs COUNT, scans the counts into
 * new offsets, allocates the exact output array, then runs WRITE.
 */
template < bool WRITE >
__global__
void two_hop_pass( SparseGraph g, node_t * new_offsets, node_t * new_neighbours )
{
    extern __shared__ unsigned bitmap[];

    int const n         = static_cast< int >( g.n );
    int const words     = ( n + 31 ) >> 5;
    int const tid       = threadIdx.x;
    int const lane      = tid & 31;
    int const warp      = tid >> 5;
    int const num_warps = blockDim.x >> 5;

    node_t const * offsets = g.neighbours_start_at;
    node_t const * nbrs    = g.neighbours;

    for( int w = tid; w < words; w += blockDim.x ) bitmap[ w ] = 0u;
    __syncthreads();

    for( int i = blockIdx.x; i < n; i += gridDim.x )
    {
        node_t const beg = offsets[ i ];
        node_t const end = offsets[ i + 1 ];
        if( beg == end ) // isolated node: no 2-hop neighbours (uniform branch for the block)
        {
            if( !WRITE && tid == 0 ) new_offsets[ i ] = 0;
            continue;
        }

        // ---- mark ----
        for( node_t p = beg + warp; p < end; p += num_warps )
        {
            node_t const k  = nbrs[ p ];
            node_t const kb = offsets[ k ];
            node_t const ke = offsets[ k + 1 ];
            for( node_t q = kb + lane; q < ke; q += 32 )
            {
                node_t const j = nbrs[ q ];
                if( j != i ) atomicOr( &bitmap[ j >> 5 ], 1u << ( j & 31 ) );
            }
        }
        __syncthreads();

        // ---- harvest ----
        if( !WRITE )
        {
            int count = 0;
            for( int w = tid; w < words; w += blockDim.x )
            {
                count += __popc( bitmap[ w ] );
                bitmap[ w ] = 0u;
            }
            int total;
            block_exclusive_scan( count, total ); // used here as a block-wide reduction
            if( tid == 0 ) new_offsets[ i ] = total;
        }
        else
        {
            node_t base = new_offsets[ i ];
            for( int w0 = 0; w0 < words; w0 += blockDim.x ) // same trip count for every thread
            {
                int const w = w0 + tid;
                unsigned bits = ( w < words ) ? bitmap[ w ] : 0u;
                int chunk_total;
                int const prefix = block_exclusive_scan( __popc( bits ), chunk_total );
                node_t * dst = new_neighbours + base + prefix;
                while( bits )
                {
                    int const b = __ffs( bits ) - 1;
                    *dst++ = ( w << 5 ) + b;
                    bits &= bits - 1;
                }
                if( w < words ) bitmap[ w ] = 0u;
                base += chunk_total;
            }
        }
        __syncthreads(); // bitmap fully cleared before the next node starts marking
    }
}

/**
 * Device buffers the sparse code needs besides the graph itself. Allocated once,
 * outside the timed region (just like the dense code's scratch matrix).
 */
struct SparseWorkspace
{
    node_t * cursor       = nullptr; /**< n     : per-row write cursors for scatter */
    node_t * unsorted     = nullptr; /**< m     : rows before sorting */
    node_t * scan_scratch = nullptr; /**< block totals for exclusive_scan */
    node_t * offsets_out  = nullptr; /**< n + 1 : offsets of the two-hop graph */

    static SparseWorkspace allocate( std::size_t n, std::size_t m )
    {
        SparseWorkspace ws;
        CUDA_CALL( cudaMalloc( (void**)&ws.cursor,       sizeof( node_t ) * std::max< std::size_t >( n, 1 ) ) );
        CUDA_CALL( cudaMalloc( (void**)&ws.unsorted,     sizeof( node_t ) * std::max< std::size_t >( m, 1 ) ) );
        CUDA_CALL( cudaMalloc( (void**)&ws.scan_scratch, sizeof( node_t ) * scan_scratch_size( n + 1 ) ) );
        CUDA_CALL( cudaMalloc( (void**)&ws.offsets_out,  sizeof( node_t ) * ( n + 1 ) ) );
        return ws;
    }

    void release()
    {
        cudaFree( cursor );
        cudaFree( unsorted );
        cudaFree( scan_scratch );
        cudaFree( offsets_out );
        cursor = unsorted = scan_scratch = offsets_out = nullptr;
    }
};

} // namespace gpu


// ---------------------------------------------------------------------------------------
// Host-side entry points
// ---------------------------------------------------------------------------------------

namespace detail {
inline int grid_for( std::size_t work, int threads )
{
    return static_cast< int >( std::max< std::size_t >( 1, std::min< std::size_t >( ( work + threads - 1 ) / threads, 1u << 16 ) ) );
}
} // namespace detail

/**
 * Builds the CSR graph from an edge list:
 *   degrees (atomics) -> exclusive scan -> scatter (atomic cursors) -> rank-sort each row.
 */
inline void build_graph( SparseGraph & g, edge_t const * d_edges, std::size_t m, gpu::SparseWorkspace const & ws )
{
    std::size_t const n = g.n;
    g.m = m;

    CUDA_CALL( cudaMemsetAsync( g.neighbours_start_at, 0, sizeof( node_t ) * ( n + 1 ) ) );
    if( m > 0 )
    {
        gpu::count_degrees<<< detail::grid_for( m, 256 ), 256 >>>( d_edges, m, g.neighbours_start_at );
        CHECK_ERROR( "count_degrees" );
    }

    gpu::exclusive_scan( g.neighbours_start_at, n + 1, ws.scan_scratch );
    if( m == 0 ) return;

    CUDA_CALL( cudaMemcpyAsync( ws.cursor, g.neighbours_start_at, sizeof( node_t ) * n, cudaMemcpyDeviceToDevice ) );
    gpu::scatter_edges<<< detail::grid_for( m, 256 ), 256 >>>( d_edges, m, ws.cursor, ws.unsorted );
    CHECK_ERROR( "scatter_edges" );
    gpu::rank_sort_rows<<< detail::grid_for( m, 256 ), 256 >>>( g, d_edges, m, ws.unsorted );
    CHECK_ERROR( "rank_sort_rows" );
}

/**
 * Replaces g with its two-hop graph (new offsets and a newly allocated neighbours array;
 * the old neighbours array is freed, the old offsets array is handed back to the workspace).
 */
inline void two_hop_reachability( SparseGraph & g, gpu::SparseWorkspace & ws )
{
    std::size_t const n = g.n;
    if( n == 0 ) return;

    // bitmap of n bits per block in shared memory
    std::size_t const smem = sizeof( unsigned ) * ( ( n + 31 ) / 32 );

    int device = 0, sms = 1, smem_optin = 48 * 1024;
    CUDA_CALL( cudaGetDevice( &device ) );
    CUDA_CALL( cudaDeviceGetAttribute( &sms, cudaDevAttrMultiProcessorCount, device ) );
    CUDA_CALL( cudaDeviceGetAttribute( &smem_optin, cudaDevAttrMaxSharedMemoryPerBlockOptin, device ) );
    if( smem > static_cast< std::size_t >( smem_optin ) )
    {
        std::fprintf( stderr, "SparseGraph two-hop: n=%zu needs a %zu-byte bitmap, device allows %d\n", n, smem, smem_optin );
        std::exit( EXIT_FAILURE );
    }
    if( smem > 48 * 1024 )
    {
        CUDA_CALL( cudaFuncSetAttribute( gpu::two_hop_pass< false >, cudaFuncAttributeMaxDynamicSharedMemorySize, static_cast< int >( smem ) ) );
        CUDA_CALL( cudaFuncSetAttribute( gpu::two_hop_pass< true  >, cudaFuncAttributeMaxDynamicSharedMemorySize, static_cast< int >( smem ) ) );
    }

    // persistent blocks: exactly as many as fit on the GPU at once
    int blocks_per_sm = 0;
    CUDA_CALL( cudaOccupancyMaxActiveBlocksPerMultiprocessor( &blocks_per_sm, gpu::two_hop_pass< false >, gpu::TWO_HOP_BLOCK, smem ) );
    int const grid = static_cast< int >( std::min< std::size_t >( n, static_cast< std::size_t >( std::max( 1, blocks_per_sm ) * sms ) ) );

    node_t * new_offsets = ws.offsets_out;

    // 1. count
    CUDA_CALL( cudaMemsetAsync( new_offsets + n, 0, sizeof( node_t ) ) );
    gpu::two_hop_pass< false ><<< grid, gpu::TWO_HOP_BLOCK, smem >>>( g, new_offsets, nullptr );
    CHECK_ERROR( "two_hop_pass<count>" );

    // 2. scan counts into offsets, read back the new edge count
    gpu::exclusive_scan( new_offsets, n + 1, ws.scan_scratch );
    node_t new_m = 0;
    CUDA_CALL( cudaMemcpy( &new_m, new_offsets + n, sizeof( node_t ), cudaMemcpyDeviceToHost ) );

    // 3. allocate exactly enough space and write
    node_t * new_neighbours = nullptr;
    CUDA_CALL( cudaMalloc( (void**)&new_neighbours, sizeof( node_t ) * std::max< std::size_t >( new_m, 1 ) ) );
    gpu::two_hop_pass< true ><<< grid, gpu::TWO_HOP_BLOCK, smem >>>( g, new_offsets, new_neighbours );
    CHECK_ERROR( "two_hop_pass<write>" );

    // 4. swap in the new graph
    CUDA_CALL( cudaFree( g.neighbours ) );
    ws.offsets_out        = g.neighbours_start_at;
    g.neighbours_start_at = new_offsets;
    g.neighbours          = new_neighbours;
    g.m                   = static_cast< std::size_t >( new_m );
}

} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "cpu_reference.h"
/**
 * Simple CPU versions of everything, used to check the GPU results.
 */
#pragma once

#include <algorithm> // std::sort
#include <cstddef>
#include <cstdint>
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2      {

/** A CSR graph in host memory (offsets has n+1 entries). */
struct CsrHost
{
    std::vector< node_t > offsets;
    std::vector< node_t > neighbours;

    bool operator==( CsrHost const & o ) const { return offsets == o.offsets && neighbours == o.neighbours; }
    std::size_t num_edges() const { return neighbours.size(); }
};

namespace cpu {

/** CSR with every row sorted. */
inline CsrHost build_csr( edge_list_t const & edges, std::size_t n )
{
    CsrHost g;
    g.offsets.assign( n + 1, 0 );
    for( auto const & e : edges ) ++g.offsets[ e.x + 1 ];
    for( std::size_t i = 0; i < n; ++i ) g.offsets[ i + 1 ] += g.offsets[ i ];
    g.neighbours.resize( edges.size() );
    std::vector< node_t > cursor( g.offsets.begin(), g.offsets.end() - 1 );
    for( auto const & e : edges ) g.neighbours[ cursor[ e.x ]++ ] = e.y;
    for( std::size_t i = 0; i < n; ++i )
        std::sort( g.neighbours.begin() + g.offsets[ i ], g.neighbours.begin() + g.offsets[ i + 1 ] );
    return g;
}

/** Row-major n x n adjacency matrix. */
inline std::vector< node_t > build_dense( edge_list_t const & edges, std::size_t n )
{
    std::vector< node_t > a( n * n, 0 );
    for( auto const & e : edges ) a[ static_cast< std::size_t >( e.x ) * n + e.y ] = 1;
    return a;
}

/** Two-hop graph via bitsets: row i = OR of rows of i's neighbours, minus i itself. */
inline CsrHost two_hop( CsrHost const & g, std::size_t n )
{
    std::size_t const words = ( n + 63 ) / 64;
    std::vector< std::uint64_t > rows( n * words, 0 );
    for( std::size_t u = 0; u < n; ++u )
        for( node_t p = g.offsets[ u ]; p < g.offsets[ u + 1 ]; ++p )
        {
            node_t const v = g.neighbours[ p ];
            rows[ u * words + v / 64 ] |= std::uint64_t( 1 ) << ( v % 64 );
        }

    CsrHost out;
    out.offsets.assign( n + 1, 0 );
    std::vector< std::uint64_t > acc( words );
    for( std::size_t i = 0; i < n; ++i )
    {
        std::fill( acc.begin(), acc.end(), 0 );
        for( node_t p = g.offsets[ i ]; p < g.offsets[ i + 1 ]; ++p )
        {
            std::uint64_t const * r = &rows[ static_cast< std::size_t >( g.neighbours[ p ] ) * words ];
            for( std::size_t w = 0; w < words; ++w ) acc[ w ] |= r[ w ];
        }
        acc[ i / 64 ] &= ~( std::uint64_t( 1 ) << ( i % 64 ) );
        for( std::size_t w = 0; w < words; ++w )
        {
            std::uint64_t bits = acc[ w ];
            while( bits )
            {
                int const b = __builtin_ctzll( bits );
                out.neighbours.push_back( static_cast< node_t >( w * 64 + b ) );
                bits &= bits - 1;
            }
        }
        out.offsets[ i + 1 ] = static_cast< node_t >( out.neighbours.size() );
    }
    return out;
}

/** Converts an adjacency matrix to sorted CSR (for comparing dense results). */
inline CsrHost dense_to_csr( std::vector< node_t > const & a, std::size_t n )
{
    CsrHost g;
    g.offsets.assign( n + 1, 0 );
    for( std::size_t i = 0; i < n; ++i )
    {
        for( std::size_t j = 0; j < n; ++j )
            if( a[ i * n + j ] ) g.neighbours.push_back( static_cast< node_t >( j ) );
        g.offsets[ i + 1 ] = static_cast< node_t >( g.neighbours.size() );
    }
    return g;
}

/** True if every cell of the matrix is 0 or 1. */
inline bool is_boolean( std::vector< node_t > const & a )
{
    for( auto x : a ) if( x != 0 && x != 1 ) return false;
    return true;
}

} // namespace cpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data, runs every GPU
 * implementation, checks each result against a CPU reference, and times them.
 *
 * Set the environment variable A2_MODE before running to choose what runs:
 *   (unset) / full : examples + correctness sweep + benchmark table
 *   quick          : examples + small correctness sweep only
 *   profile        : one medium-sized graph only (use this under ncu / nsys)
 */

#include <cstdio>
#include <cstdlib>  // std::getenv
#include <iostream> // std::cout, std::endl
#include <string>
#include <vector>

#include "dense_graph.h"
#include "sparse_graph.h"

#include "cpu_reference.h"
#include "data_generator.h"
#include "data_types.h"

namespace {

using namespace csc485b;
using a2::edge_list_t;
using a2::edge_t;
using a2::node_t;

/** Times work on the GPU with CUDA events. */
struct GpuTimer
{
    cudaEvent_t start_, stop_;
    GpuTimer()  { cudaEventCreate( &start_ ); cudaEventCreate( &stop_ ); }
    ~GpuTimer() { cudaEventDestroy( start_ ); cudaEventDestroy( stop_ ); }
    void start() { cudaEventRecord( start_ ); }
    float stop()
    {
        cudaEventRecord( stop_ );
        cudaEventSynchronize( stop_ );
        float ms = 0.f;
        cudaEventElapsedTime( &ms, start_, stop_ );
        return ms;
    }
};

struct Timings { float build_ms = 0.f, reach_ms = 0.f; };

/** Allocates a dense graph, builds it, runs two-hop; optionally copies both results back. */
Timings run_dense( edge_t const * d_edges, std::size_t n, std::size_t m, a2::DenseAlgo algo,
                   std::vector< node_t > * built, std::vector< node_t > * reach )
{
    node_t * d_matrix, * d_scratch;
    CUDA_CALL( cudaMalloc( (void**)&d_matrix,  sizeof( node_t ) * n * n ) );
    CUDA_CALL( cudaMalloc( (void**)&d_scratch, sizeof( node_t ) * n * n ) );
    a2::DenseGraph g{ n, d_matrix };

    GpuTimer timer;
    Timings t;

    CUDA_CALL( cudaDeviceSynchronize() );
    timer.start();
    a2::build_graph( g, d_edges, m );
    t.build_ms = timer.stop();

    if( built )
    {
        built->resize( g.matrix_size() );
        CUDA_CALL( cudaMemcpy( built->data(), g.adjacencyMatrix, sizeof( node_t ) * g.matrix_size(), cudaMemcpyDeviceToHost ) );
    }

    timer.start();
    a2::two_hop_reachability( g, d_scratch, algo );
    t.reach_ms = timer.stop();

    if( reach )
    {
        reach->resize( g.matrix_size() );
        CUDA_CALL( cudaMemcpy( reach->data(), g.adjacencyMatrix, sizeof( node_t ) * g.matrix_size(), cudaMemcpyDeviceToHost ) );
    }

    CUDA_CALL( cudaFree( g.adjacencyMatrix ) );
    CUDA_CALL( cudaFree( d_scratch ) );
    return t;
}

a2::CsrHost download( a2::SparseGraph const & g )
{
    a2::CsrHost h;
    h.offsets.resize( g.n + 1 );
    h.neighbours.resize( g.m );
    CUDA_CALL( cudaMemcpy( h.offsets.data(), g.neighbours_start_at, sizeof( node_t ) * ( g.n + 1 ), cudaMemcpyDeviceToHost ) );
    if( g.m > 0 )
        CUDA_CALL( cudaMemcpy( h.neighbours.data(), g.neighbours, sizeof( node_t ) * g.m, cudaMemcpyDeviceToHost ) );
    return h;
}

/** Allocates a sparse graph, builds it, runs two-hop; optionally copies both results back. */
Timings run_sparse( edge_t const * d_edges, std::size_t n, std::size_t m,
                    a2::CsrHost * built, a2::CsrHost * reach )
{
    node_t * d_offsets, * d_neighbours;
    CUDA_CALL( cudaMalloc( (void**)&d_offsets,    sizeof( node_t ) * ( n + 1 ) ) );
    CUDA_CALL( cudaMalloc( (void**)&d_neighbours, sizeof( node_t ) * std::max< std::size_t >( m, 1 ) ) );
    a2::SparseGraph g{ n, m, d_offsets, d_neighbours };
    auto ws = a2::gpu::SparseWorkspace::allocate( n, m );

    GpuTimer timer;
    Timings t;

    CUDA_CALL( cudaDeviceSynchronize() );
    timer.start();
    a2::build_graph( g, d_edges, m, ws );
    t.build_ms = timer.stop();

    if( built ) *built = download( g );

    timer.start();
    a2::two_hop_reachability( g, ws );
    t.reach_ms = timer.stop();

    if( reach ) *reach = download( g );

    CUDA_CALL( cudaFree( g.neighbours_start_at ) );
    CUDA_CALL( cudaFree( g.neighbours ) );
    ws.release();
    return t;
}

// ----------------------------------------------------------------------------------------
// printing helpers for small graphs
// ----------------------------------------------------------------------------------------

void print_matrix( char const * label, std::vector< node_t > const & a, std::size_t n )
{
    std::cout << label << ":\n";
    for( std::size_t i = 0; i < n; ++i )
    {
        std::cout << "  ";
        for( std::size_t j = 0; j < n; ++j ) std::cout << a[ i * n + j ] << ' ';
        std::cout << '\n';
    }
}

void print_csr( char const * label, a2::CsrHost const & g )
{
    std::cout << label << ":\n  neighbours_start_at: [";
    for( std::size_t i = 0; i + 1 < g.offsets.size(); ++i ) std::cout << ( i ? ", " : "" ) << g.offsets[ i ];
    std::cout << "]   (+ end marker " << g.offsets.back() << ")\n  neighbours: [";
    for( std::size_t i = 0; i < g.neighbours.size(); ++i ) std::cout << ( i ? ", " : "" ) << g.neighbours[ i ];
    std::cout << "]\n";
}

// ----------------------------------------------------------------------------------------
// one complete test: GPU dense (tiled) + GPU sparse, checked against the CPU
// ----------------------------------------------------------------------------------------

struct TestResult
{
    bool ok = true;
    Timings dense, sparse;
    std::size_t two_hop_edges = 0;
};

TestResult test_graph( edge_list_t const & edges, std::size_t n, bool verbose )
{
    std::size_t const m = edges.size();
    TestResult r;

    edge_t * d_edges = nullptr;
    CUDA_CALL( cudaMalloc( (void**)&d_edges, sizeof( edge_t ) * std::max< std::size_t >( m, 1 ) ) );
    if( m > 0 )
        CUDA_CALL( cudaMemcpy( d_edges, edges.data(), sizeof( edge_t ) * m, cudaMemcpyHostToDevice ) );

    std::vector< node_t > dense_built, dense_reach;
    a2::CsrHost sparse_built, sparse_reach;
    r.dense  = run_dense ( d_edges, n, m, a2::DenseAlgo::Tiled, &dense_built, &dense_reach );
    r.sparse = run_sparse( d_edges, n, m, &sparse_built, &sparse_reach );
    CUDA_CALL( cudaFree( d_edges ) );

    // CPU reference
    a2::CsrHost const ref_built = a2::cpu::build_csr( edges, n );
    a2::CsrHost const ref_reach = a2::cpu::two_hop( ref_built, n );
    r.two_hop_edges = ref_reach.num_edges();

    bool const dense_build_ok  = dense_built == a2::cpu::build_dense( edges, n );
    bool const dense_reach_ok  = a2::cpu::is_boolean( dense_reach ) && a2::cpu::dense_to_csr( dense_reach, n ) == ref_reach;
    bool const sparse_build_ok = sparse_built == ref_built;
    bool const sparse_reach_ok = sparse_reach == ref_reach;
    r.ok = dense_build_ok && dense_reach_ok && sparse_build_ok && sparse_reach_ok;

    if( verbose )
    {
        print_matrix( "Dense build", dense_built, n );
        print_csr( "Sparse build", sparse_built );
        print_matrix( "Dense two-hop", dense_reach, n );
        print_csr( "Sparse two-hop", sparse_reach );
    }
    if( !r.ok )
    {
        std::cout << "  !! MISMATCH  dense build " << ( dense_build_ok ? "ok" : "WRONG" )
                  << ", dense two-hop "  << ( dense_reach_ok  ? "ok" : "WRONG" )
                  << ", sparse build "   << ( sparse_build_ok ? "ok" : "WRONG" )
                  << ", sparse two-hop " << ( sparse_reach_ok ? "ok" : "WRONG" ) << std::endl;
    }
    return r;
}

/** Erdős-Rényi graph where `density` is the fraction of non-zero adjacency-matrix cells. */
edge_list_t random_graph( std::size_t n, double density )
{
    std::size_t const m = static_cast< std::size_t >( density * static_cast< double >( n * ( n - 1 ) ) );
    return a2::generate_graph( n, std::min( m, n * ( n - 1 ) - 1 ) );
}

/** Checks the naive dense kernel against the tiled one on a graph. */
bool check_naive( edge_list_t const & edges, std::size_t n, float * naive_ms )
{
    std::size_t const m = edges.size();
    edge_t * d_edges;
    CUDA_CALL( cudaMalloc( (void**)&d_edges, sizeof( edge_t ) * std::max< std::size_t >( m, 1 ) ) );
    if( m > 0 ) CUDA_CALL( cudaMemcpy( d_edges, edges.data(), sizeof( edge_t ) * m, cudaMemcpyHostToDevice ) );
    std::vector< node_t > naive, tiled;
    Timings const t = run_dense( d_edges, n, m, a2::DenseAlgo::Naive, nullptr, &naive );
    run_dense( d_edges, n, m, a2::DenseAlgo::Tiled, nullptr, &tiled );
    CUDA_CALL( cudaFree( d_edges ) );
    if( naive_ms ) *naive_ms = t.reach_ms;
    return naive == tiled;
}

} // namespace


int main()
{
    std::string const mode = std::getenv( "A2_MODE" ) ? std::getenv( "A2_MODE" ) : "full";

    cudaDeviceProp prop;
    CUDA_CALL( cudaGetDeviceProperties( &prop, 0 ) );
    std::cout << "Device: " << prop.name << " (" << prop.multiProcessorCount << " SMs), mode = " << mode << "\n\n";
    CUDA_CALL( cudaFree( 0 ) ); // create the context up front so it isn't timed

    bool all_ok = true;

    if( mode == "profile" )
    {
        std::size_t const n = 4096;
        auto const edges = random_graph( n, 0.01 );
        auto const r = test_graph( edges, n, false );
        std::printf( "profile run n=%zu m=%zu: dense %.3f ms, sparse %.3f ms, %s\n",
                     n, edges.size(), r.dense.reach_ms, r.sparse.reach_ms, r.ok ? "PASS" : "FAIL" );
        return r.ok ? EXIT_SUCCESS : EXIT_FAILURE;
    }

    // ---- 1. the example from the assignment handout ----
    {
        std::cout << "=== Handout example: edges (0,1), (0,2), (2,3) ===\n";
        edge_list_t const edges = { make_int2( 0, 1 ), make_int2( 1, 0 ), make_int2( 0, 2 ),
                                    make_int2( 2, 0 ), make_int2( 2, 3 ), make_int2( 3, 2 ) };
        auto const r = test_graph( edges, 4, true );
        std::cout << "Expected two-hop CSR: neighbours_start_at: [0, 1, 2, 3]  neighbours: [3, 2, 1, 0]\n";
        std::cout << ( r.ok ? "PASS" : "FAIL" ) << "\n\n";
        all_ok &= r.ok;
    }

    // ---- 2. the starter code's random n = 8 graph ----
    {
        std::size_t constexpr n = 8;
        std::size_t constexpr expected_degree = n >> 1;
        auto const edges = a2::generate_graph( n, n * expected_degree );
        std::cout << "=== Random graph, n = 8 ===\nInput: ";
        for( auto const & e : edges ) std::cout << "(" << e.x << "," << e.y << ") ";
        std::cout << "\n";
        auto const r = test_graph( edges, n, true );
        std::cout << ( r.ok ? "PASS" : "FAIL" ) << "\n\n";
        all_ok &= r.ok;
    }

    // ---- 3. correctness sweep (odd sizes on purpose, to hit partial tiles / partial warps) ----
    {
        std::cout << "=== Correctness sweep (GPU vs CPU reference) ===\n";
        struct Case { std::size_t n; double density; };
        std::vector< Case > cases = { { 2, 0.6 }, { 5, 0.5 }, { 31, 0.1 }, { 33, 0.3 }, { 64, 0.97 },
                                      { 100, 0.0 }, { 100, 0.05 }, { 129, 0.5 }, { 257, 0.02 } };
        if( mode != "quick" )
            cases.insert( cases.end(), { { 1000, 0.01 }, { 1000, 0.2 }, { 1500, 0.003 }, { 2047, 0.05 }, { 5000, 0.001 } } );

        for( auto const & c : cases )
        {
            auto const edges = random_graph( c.n, c.density );
            auto const r = test_graph( edges, c.n, false );
            bool const naive_ok = c.n > 2100 || check_naive( edges, c.n, nullptr );
            bool const ok = r.ok && naive_ok;
            std::printf( "  n=%5zu density=%6.3f m=%8zu two-hop edges=%9zu  %s%s\n", c.n, c.density, edges.size(),
                         r.two_hop_edges, ok ? "PASS" : "FAIL", naive_ok ? "" : " (naive dense differs)" );
            all_ok &= ok;
        }
        std::cout << "\n";
    }

    if( mode == "quick" )
    {
        std::cout << ( all_ok ? "ALL TESTS PASSED" : "SOME TESTS FAILED" ) << std::endl;
        return all_ok ? EXIT_SUCCESS : EXIT_FAILURE;
    }

    // ---- 4. benchmark ----
    {
        std::cout << "=== Benchmark (times in ms; every row is also checked against the CPU) ===\n";
        std::vector< std::size_t > const sizes     = { 2048, 4096, 8192 };
        std::vector< double >      const densities = { 0.001, 0.005, 0.01, 0.025, 0.05, 0.10 };

        for( auto const n : sizes )
        {
            float naive_ms = 0.f;
            if( n <= 4096 )
            {
                bool const ok = check_naive( random_graph( n, 0.01 ), n, &naive_ms );
                std::printf( "\nn = %zu   naive dense two-hop baseline: %.3f ms %s\n", n, naive_ms, ok ? "" : "(MISMATCH)" );
                all_ok &= ok;
            }
            else
            {
                std::printf( "\nn = %zu   (naive dense baseline skipped: too slow)\n", n );
            }
            std::printf( "  %8s %10s %11s | %11s %11s | %11s %11s | %8s  %s\n", "density", "m", "2hop edges",
                         "dense bld", "sparse bld", "dense 2hop", "sparse 2hop", "speedup", "check" );

            for( auto const density : densities )
            {
                auto const edges = random_graph( n, density );
                auto const r = test_graph( edges, n, false );
                std::printf( "  %7.2f%% %10zu %11zu | %11.3f %11.3f | %11.3f %11.3f | %7.2fx  %s\n",
                             100.0 * density, edges.size(), r.two_hop_edges,
                             r.dense.build_ms, r.sparse.build_ms, r.dense.reach_ms, r.sparse.reach_ms,
                             r.dense.reach_ms / r.sparse.reach_ms, r.ok ? "PASS" : "FAIL" );
                all_ok &= r.ok;
            }
        }
        std::cout << "\n(speedup = dense two-hop time / sparse two-hop time; > 1 means CSR is faster)\n\n";
    }

    std::cout << ( all_ok ? "ALL TESTS PASSED" : "SOME TESTS FAILED" ) << std::endl;
    return all_ok ? EXIT_SUCCESS : EXIT_FAILURE;
}

In [ ]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -lineinfo -std=c++20 -arch=sm_75"